# Pipeline

The pipeline automatically runs the 3 basic processing steps:

- finds objects on the first plate, that have no corresponding object on the second plate;
- compares between objects with no match, with objects that have a match;
- displays the non-matched objects that have a higher likelihood of being a "real event" (and 
  not a plate artifact).
  
The sequence is defined by an entry in the *sequences* dictionary in file *settings.py*.
These are sequences of plate ID numbers. Existing sequences currently in the file were produced
by notebook *footoprints.ipynb*.

Each step of the pipelin is run by a separate notebook, with results dumped in subdirectory 
**./html/**. Results have the same look as the corresponding input notebook run for a 
particular pair of plates, but are read-only, and formatted as an HTML web page. 

The notebooks can be run individually in interactive form. The input for a notebook is controlled 
by the content of file *dataset.json*, which defines the pair of plates one wants to study. This file
is rewritten by the pipeline, so make sure it points to the correct pair of plates before running 
any notebook (pay attention to the spelling, it's a JSON file which has stringent formatting 
requirements).

The pipelne requires that all data be previously installed in the data directory (defined in file
*settings.py*). The data can be automatically downloaded by script *download.ipynb*. This script is
also driven by the sequences in *settings.py*.

In [1]:
import os
import json
from importlib import reload

import settings
from settings import DATAPATH, RESULTS, tel_suffix, sequences, current_sequence, current_dataset
from library import update_dataset, update_sequence

In [2]:
# output paths
output_path = os.path.join(DATAPATH, "html")
output_path_results = RESULTS

In [3]:
def run_pipeline(seq_key):

    update_sequence(seq_key)
    reload(settings)
    from settings import current_sequence
    
    sequence = sequences[current_sequence]
    
    print("START pipeline for sequence ", tel_suffix, seq_key, " ", sequence)
    
    for i in range(len((sequence)) - 1):

        plate_id_str = str(sequence[i])
        next_plate_id_str = str(sequence[i+1])

        key = plate_id_str + ',' + next_plate_id_str
        print("START processing dataset: ", key)

        update_dataset(key)
        reload(settings)
        from settings import current_dataset

        suffix = plate_id_str + "_" + next_plate_id_str + ".html"

        try:
            # to perform PSF analysis and display operations, this step can be removed in
            # order to expedite execution. It is only required when sextractor-related
            # filtering has to be re-done on the input tables.
            filename = os.path.join(output_path, "find_mismatches_" + suffix)
            !jupyter nbconvert --to html --execute find_mismatches.ipynb --output $filename

            filename = os.path.join(output_path, "psf_analysis_" + suffix)
            !jupyter nbconvert --to html --execute psf_analysis.ipynb --output $filename

            filename = os.path.join(output_path, "display_nomatches_" + suffix)
            !jupyter nbconvert --to html --execute display_nonmatches.ipynb --output $filename

        except Exception as e:
            print("--------  ERROR: ", str(e))
            print()
            continue

        print("END processing dataset: ", key)

    print("Collating results...")

    suffix = tel_suffix + "_" + str(seq_key) + ".html"

    filename = os.path.join(output_path, "pipeline_final_" + suffix)
    !jupyter nbconvert --to html --execute collate.ipynb --output $filename

    print("Printing results file...")

    filename = os.path.join(output_path_results, "pipeline_view_results_" + suffix)
    !jupyter nbconvert --to html --execute pipeline_results.ipynb --output $filename

    print("END pipeline for sequence ", tel_suffix, " ", seq_key)

In [4]:
# # run pipeline on any given sequence: set current_sequence in dataset.json
# run_pipeline(current_sequence)

In [ ]:
# alternatively, run pipeline on all sequences defined in import_xx.py

keys = list(sequences.keys())
print(keys)

for seq_key in keys:
    
    run_pipeline(seq_key)

['seq94', 'seq96', 'seq98', 'seq102', 'seq103', 'seq104', 'seq119', 'seq122', 'seq137', 'seq138']
START pipeline for sequence  DR seq94   [62163, 62162]
START processing dataset:  62163,62162
[NbConvertApp] Converting notebook find_mismatches.ipynb to html
w1  -  12% .  2000 40661710009257
[NbConvertApp] Writing 321873 bytes to /Users/busko/Projects/VASCO_data/footprints/html/find_mismatches_62163_62162.html
[NbConvertApp] Converting notebook psf_analysis.ipynb to html
FitWorker  w0  - started.
FitWorker  w1  - started.
FitWorker  w0  - ended.
FitWorker  w2  - started.
FitWorker  w1  - ended.
FitWorker  w3  - started.
FitWorker  w2  - ended.
FitWorker  w3  - ended.
FitWorker  w0  - started.
FitWorker  w1  - started.
FitWorker  w0  - ended.
FitWorker  w2  - started.
FitWorker  w1  - ended.
FitWorker  w3  - started.
FitWorker  w2  - ended.
FitWorker  w3  - ended.
w0  -  0% .   0
w0  -  85% .   100
w1  -  0% .   0
w1  -  85% .   100
w2  -  0% .   0
w2  -  85% .   100
w3  -  0% .   0
w3  -